# Week 2: roofline analysis for T4 and H100

This notebook walks through the roofline model — the single most useful mental model for whether an LLM workload is bottlenecked by compute (FLOPs/s) or memory bandwidth (bytes/s read from HBM). Everything here is **modelled**: equations and public spec sheets, no measurement. The T4 numbers are later checked against actual vLLM runs in `03_vllm_t4_validation.ipynb`.

## Why roofline

The achievable throughput of any compute kernel is upper-bounded by:

$$\text{achieved FLOPs/s} \le \min\left(\text{peak FLOPs/s},\ \text{peak bandwidth} \times \text{arithmetic intensity}\right)$$

where **arithmetic intensity** is FLOPs performed per byte moved from HBM.

Three regimes:
- **Low intensity** — bandwidth ceiling binds. Kernel is **memory-bound**. Adding more compute wastes silicon.
- **At the knee** — both ceilings equal. Arithmetic intensity $= $ peak FLOPs / peak bandwidth.
- **High intensity** — compute ceiling binds. Kernel is **compute-bound**. Only more FLOPs help.

For LLM inference the regimes map cleanly:
- **Prefill** — large batch × long input → high intensity → compute-bound.
- **Decode** — batch=1 single-token step → intensity ~ 2 → memory-bound. This is why decode is **slow** and **HBM bandwidth matters more than FP16 TFLOPS** for latency-sensitive serving.

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import plotly.graph_objects as go
from core.gpu_specs import load_catalog
from core.cost_engine.kv_cache import LLAMA3_8B, LLAMA3_70B, TINYLLAMA_1_1B, QWEN2_5_3B
from core.cost_engine.roofline import decode_tps, prefill_tps, roofline_tflops

catalog = load_catalog()
t4 = catalog.by_id('t4')
h100 = catalog.by_id('h100_sxm5_80gb')
print(f'T4  : {t4.fp16_tflops_dense:>6.0f} TFLOPS FP16, {t4.memory_bandwidth_gbs:>5.0f} GB/s HBM')
print(f'H100: {h100.fp16_tflops_dense:>6.0f} TFLOPS FP16, {h100.memory_bandwidth_gbs:>5.0f} GB/s HBM')
print(f'Knee FLOPs/byte: T4 {t4.fp16_tflops_dense*1e12/t4.memory_bandwidth_gbs/1e9:.1f}, H100 {h100.fp16_tflops_dense*1e12/h100.memory_bandwidth_gbs/1e9:.1f}')

The knee is roughly 200 FLOPs/byte for T4 and 300 FLOPs/byte for H100. **Any kernel with arithmetic intensity below the knee is memory-bound on that GPU.** Decode steps live at intensity ~ 2-10. There is no amount of smarter kernel work that moves a batch-1 decode off the memory ceiling.

In [ ]:
intensities = np.logspace(0, 4, 50)

def curve(gpu, util=0.65):
    return [roofline_tflops(gpu, float(i), util)[0] for i in intensities]

fig = go.Figure()
fig.add_trace(go.Scatter(x=intensities, y=curve(t4), mode='lines', name='T4 (65% util)'))
fig.add_trace(go.Scatter(x=intensities, y=curve(h100), mode='lines', name='H100 SXM5 (65% util)'))
fig.update_layout(
    xaxis_type='log', yaxis_type='log',
    xaxis_title='Arithmetic intensity (FLOPs / byte)',
    yaxis_title='Achievable TFLOPS',
    title='Roofline — T4 vs H100 (FP16 tensor cores, 65% kernel utilisation)',
    height=500,
)
fig.show()

## Decode is always memory-bound at serving batches

Decode step reads the full weight tensor once per step, does $O(B \cdot P)$ FLOPs for batch $B$ and $P$ parameters. Intensity $\approx 2B$ for the GEMM plus a KV-cache read. Even at $B=64$ this is ~130 FLOPs/byte — still below the H100 knee.

Consequence: **H100 decode throughput is roughly `HBM_bandwidth * utilisation / weight_bytes`** at small batches, with batching multiplying aggregate TPS until the knee.

In [ ]:
models = [('TinyLlama-1.1B', TINYLLAMA_1_1B, 1.1), ('Qwen2.5-3B', QWEN2_5_3B, 3.0), ('Llama-3-8B', LLAMA3_8B, 8.0), ('Llama-3-70B', LLAMA3_70B, 70.0)]
batches = [1, 2, 4, 8, 16, 32, 64, 128]

print(f"{'Model':<16} {'GPU':<8} {'B=1':>7} {'B=8':>7} {'B=32':>7} {'B=128':>7}")
for name, arch, p in models:
    for gpu in [t4, h100]:
        if name == 'Llama-3-70B' and gpu.id == 't4':
            continue  # 70B does not fit on T4
        tps = [decode_tps(gpu, arch, p, batch_size=b, kv_seq_len=1024).tokens_per_second for b in batches]
        print(f'{name:<16} {gpu.id:<8} {tps[0]:>7.0f} {tps[batches.index(8)]:>7.0f} {tps[batches.index(32)]:>7.0f} {tps[batches.index(128)]:>7.0f}')

## The 70B-on-H100 headline

For the demo angle (Nscale/Nebius neocloud unit economics), the headline number is: *what decode TPS can one H100 SXM5 produce running Llama-3-70B FP16, batch 32?*

Spoiler: 70B FP16 weights (140 GB) do not fit on one H100 (80 GB). So the realistic scenarios are:
- **FP8** — 70 GB weights, fits. Decode intensity still ~ 20-60. Memory-bound.
- **INT4** — 35 GB weights, fits with large KV room. Decode still memory-bound but ~2x the FP8 speed since weight bandwidth halves.
- **Sharded FP16** — 2x H100 NVLink. Not covered in this calculator (scope: single-GPU).

In [ ]:
# Headline demo numbers
scenarios = [
    ('Llama-3-70B FP8 batch=32',  70.0, 'fp8',  32),
    ('Llama-3-70B INT4 batch=32', 70.0, 'int4', 32),
    ('Llama-3-8B  FP16 batch=32',  8.0, 'fp16', 32),
]
for name, p, dtype, b in scenarios:
    arch = LLAMA3_70B if 'Llama-3-70B' in name else LLAMA3_8B
    res = decode_tps(h100, arch, p, batch_size=b, kv_seq_len=1024, dtype=dtype)
    print(f'{name:<30} {res.tokens_per_second:>7.0f} tok/s ({res.bound})')

## What this changes about procurement

If decode is memory-bound at your workload, three decisions that look technical are actually procurement decisions:

1. **Pick H200 over H100.** Same compute; HBM3e lifts bandwidth from 3.35 TB/s to 4.8 TB/s. That is roughly a **40% decode speedup** at identical capex. The $/hr markup should be well under 40% for H200 to pay off.
2. **Quantise before you buy a bigger GPU.** FP16 → FP8 halves the bytes streamed per decode step. The speedup is model- and kernel-dependent but typically 1.6-1.9x — more cost-effective than any GPU upgrade on the same bandwidth tier.
3. **B200/B300/R100 claims are mostly about FP4 and sparsity.** FP16 dense TFLOPS roughly doubles generation-to-generation, but **HBM3e bandwidth** scales more slowly (8 TB/s on B200). Decode speed improvements per generation are smaller than headline compute numbers suggest. This matters for anyone selling capacity as $/M tokens.

Week 2 writes this chart into the Streamlit Validation page. Week 3 wires the error bar in once real T4 runs land.